# Capstone: Deployment Control Center

This capstone ties together Chapters 1 to 6 into one realistic story.

We are building a small deployment control center for internal platform work. The goal is not to build a huge production system in one notebook. The goal is to understand how advanced Python pieces fit together in a practical tool.

This project combines:
- Chapter 1: dataclasses, decorators, protocols, context managers
- Chapter 2: asyncio and bounded concurrency
- Chapter 3: API client patterns and retry thinking
- Chapter 4: Pydantic validation and settings
- Chapter 5: FastAPI service structure
- Chapter 6: SQLAlchemy async persistence


## 1. Capstone Problem Statement

Imagine you are building an internal platform tool that:
- receives deployment requests
- validates them
- fans out deployment actions to multiple services
- talks to external systems such as config or health endpoints
- records deployment state in a database
- exposes an API for operators or CI pipelines

That is a very realistic DevOps-flavored Python problem.

## 2. Architecture at a Glance

The flow is:

1. FastAPI route receives a request
2. Pydantic validates the payload and settings
3. Service layer orchestrates the deployment
4. Async deployment client performs concurrent health checks / rollout actions
5. SQLAlchemy persists deployment state
6. API returns a clean response model

The code below focuses on the core engineering ideas, not on every production detail.

In [ ]:
from __future__ import annotations

import asyncio
import random
import time
import uuid
from contextlib import contextmanager
from dataclasses import dataclass, field
from enum import Enum
from functools import wraps
from typing import Protocol

from pydantic import BaseModel, Field, ConfigDict, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

## 3. Pydantic Models and Settings

We start with validated inputs and typed settings. This is the contract layer of the system.

In [ ]:
class Environment(str, Enum):
    DEV = 'dev'
    STAGING = 'staging'
    PROD = 'prod'


class DeploymentRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')

    service_name: str = Field(..., pattern=r'^[a-z][a-z0-9-]{1,49}$')
    image_tag: str = Field(..., min_length=1)
    environment: Environment
    replicas: int = Field(default=1, ge=1, le=20)
    requested_by: str = Field(..., min_length=1)


class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix='CONTROL_', extra='ignore')

    api_name: str = 'deployment-control-center'
    max_concurrency: int = 3
    default_timeout_seconds: int = 5
    auth_token: SecretStr = SecretStr('demo-token')


settings = AppSettings()
request = DeploymentRequest(
    service_name='auth-api',
    image_tag='v2.1.0',
    environment=Environment.PROD,
    replicas=3,
    requested_by='ci-pipeline',
)

print(settings)
print(request.model_dump())

## 4. Domain Objects, Protocols, Decorators, and Context Managers

This section brings in Chapter 1 ideas. We use a dataclass for internal state, a protocol for pluggable deploy backends, a retry decorator, and a timer context manager.

In [ ]:
@dataclass
class DeploymentRecord:
    id: str
    service_name: str
    image_tag: str
    environment: Environment
    replicas: int
    requested_by: str
    status: str = 'pending'
    events: list[str] = field(default_factory=list)


class DeployBackend(Protocol):
    async def deploy(self, record: DeploymentRecord) -> dict:
        ...


def retry_async(max_attempts: int = 3, delay: float = 0.1):
    def decorator(fn):
        @wraps(fn)
        async def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return await fn(*args, **kwargs)
                except Exception as exc:
                    last_exc = exc
                    if attempt < max_attempts:
                        await asyncio.sleep(delay * attempt)
            raise last_exc
        return wrapper
    return decorator


@contextmanager
def timed_block(label: str):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f'{label}: {elapsed:.3f}s')

## 5. Async Deployment Client with Bounded Concurrency

This section brings in Chapters 2 and 3. We simulate an async deployment backend with concurrency control and retry behavior.

In [ ]:
class SimulatedDeployClient:
    def __init__(self, max_concurrency: int = 3):
        self.semaphore = asyncio.Semaphore(max_concurrency)

    @retry_async(max_attempts=3, delay=0.05)
    async def deploy(self, record: DeploymentRecord) -> dict:
        async with self.semaphore:
            await asyncio.sleep(random.uniform(0.1, 0.3))
            if random.random() < 0.15:
                raise ConnectionError(f'temporary failure for {record.service_name}')
            return {
                'deployment_id': record.id,
                'service_name': record.service_name,
                'status': 'running',
                'environment': record.environment.value,
            }


client = SimulatedDeployClient(max_concurrency=settings.max_concurrency)

## 6. Orchestration Layer

This is the service layer. It converts validated requests into internal records, calls the backend, and updates deployment state.

In [ ]:
async def run_deployment(req: DeploymentRequest, backend: DeployBackend) -> DeploymentRecord:
    record = DeploymentRecord(
        id=str(uuid.uuid4()),
        service_name=req.service_name,
        image_tag=req.image_tag,
        environment=req.environment,
        replicas=req.replicas,
        requested_by=req.requested_by,
    )
    record.events.append('request_validated')

    result = await backend.deploy(record)
    record.status = result['status']
    record.events.append('deployment_started')
    return record


with timed_block('single deployment'):
    random.seed(7)
    record = asyncio.run(run_deployment(request, client))

print(record)

## 7. Multi-Service Fan-Out

A very common pattern in internal platform tooling is to fan out the same operation to many services or environments.

In [ ]:
async def rollout_many(service_names: list[str]) -> list[DeploymentRecord]:
    requests = [
        DeploymentRequest(
            service_name=name,
            image_tag='v9.9.9',
            environment=Environment.STAGING,
            replicas=2,
            requested_by='ops-bot',
        )
        for name in service_names
    ]
    tasks = [run_deployment(req, client) for req in requests]
    return await asyncio.gather(*tasks)


random.seed(21)
records = asyncio.run(rollout_many(['auth-api', 'user-svc', 'payment-api', 'analytics']))
for item in records:
    print(item.service_name, item.status, item.events)

## 8. FastAPI Layer

Below is a compact FastAPI shape for the capstone. This is intentionally shown as a code blueprint rather than executed in the notebook.

In [ ]:
FASTAPI_BLUEPRINT = """
from fastapi import FastAPI, Depends
from typing import Annotated

app = FastAPI(title='Deployment Control Center')

def get_settings() -> AppSettings:
    return AppSettings()

@app.post('/deployments')
async def create_deployment(
    req: DeploymentRequest,
    settings: Annotated[AppSettings, Depends(get_settings)],
):
    backend = SimulatedDeployClient(max_concurrency=settings.max_concurrency)
    record = await run_deployment(req, backend)
    return record
"""

print(FASTAPI_BLUEPRINT)

## 9. SQLAlchemy Async Layer

This is the database shape that would persist deployment records in a real service.

In [ ]:
SQLALCHEMY_BLUEPRINT = """
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import String, Integer, DateTime
from datetime import datetime

class Base(DeclarativeBase):
    pass

class DeploymentRow(Base):
    __tablename__ = 'deployments'
    id: Mapped[str] = mapped_column(String(36), primary_key=True)
    service_name: Mapped[str] = mapped_column(String(100))
    image_tag: Mapped[str] = mapped_column(String(100))
    environment: Mapped[str] = mapped_column(String(20))
    replicas: Mapped[int] = mapped_column(Integer)
    status: Mapped[str] = mapped_column(String(20))
    requested_by: Mapped[str] = mapped_column(String(100))
    created_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.utcnow)
"""

print(SQLALCHEMY_BLUEPRINT)

## 10. What This Capstone Teaches

This notebook is worth revisiting because it teaches how the pieces fit together, not just what each library does in isolation.

If you can explain this capstone clearly, you can tell a strong interview story:
- Pydantic validates the boundary
- dataclasses and protocols shape internal design
- decorators and context managers add cross-cutting behavior cleanly
- asyncio gives safe fan-out with bounded concurrency
- FastAPI exposes the service cleanly
- SQLAlchemy gives persistence and transaction boundaries

## 11. Next Practical Extensions

Natural next improvements would be:
- replace the simulated client with real `httpx.AsyncClient`
- store records in a real async SQLAlchemy database
- add a request ID middleware in FastAPI
- add pytest tests for the service layer and API routes
- package it as a real internal CLI plus service

That is exactly how toy knowledge becomes tool-building skill.